## Functions

In [ ]:
import os
import re
import json
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 14,          # 默认字体大小
    "axes.titlesize": 14,     # title
    "axes.labelsize": 14,     # x/y label
    "xtick.labelsize": 14,    # x tick
    "ytick.labelsize": 14,    # y tick
    "legend.fontsize": 14,    # legend
})


def _sanitize_filename(s: str) -> str:
    """
    Make a string safe for filenames (keep letters, numbers, _, -, .).
    """
    s = str(s)
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", s)
    return s


def _load_results_json(path: str) -> Dict[str, List[float]]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"results.json not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        raise ValueError(f"Invalid results.json format (must be dict): {path}")

    # Ensure all values are lists
    out = {}
    for k, v in obj.items():
        if not isinstance(v, list):
            raise ValueError(f"Metric '{k}' value must be a list in {path}, got: {type(v)}")
        out[k] = v
    return out


def _validate_lengths(results: Dict[str, List[float]], exp_name: str, path: str) -> int:
    """
    Validate all metric lists have equal length inside one experiment.
    Return the run count.
    """
    lengths = {k: len(v) for k, v in results.items()}
    if len(set(lengths.values())) != 1:
        # Show a concise diagnostic
        items = sorted(lengths.items(), key=lambda x: x[0])
        msg = ", ".join([f"{k}:{n}" for k, n in items[:10]])
        raise ValueError(
            f"[{exp_name}] Inconsistent run counts across metrics in {path}. "
            f"Examples: {msg} ..."
        )
    return next(iter(lengths.values())) if lengths else 0


def _get_filter_key(filter_cfg: Dict[str, Any]) -> str:
    # You said filter dict uses 'filter_key', but we also accept 'key' defensively.
    if "filter_key" in filter_cfg:
        return filter_cfg["filter_key"]
    if "key" in filter_cfg:
        return filter_cfg["key"]
    raise KeyError("filter must contain 'filter_key' (or 'key').")


def _select_run_indices(
    results: Dict[str, List[float]],
    exp_name: str,
    filter_cfg: Optional[Dict[str, Any]],
) -> List[int]:
    """
    Return selected run indices according to filter_cfg.
    Also print unselected indices + filter values + selected ratio.
    """
    if not results:
        print(f"[{exp_name}] Empty results.")
        return []

    any_metric = next(iter(results.values()))
    n_runs = len(any_metric)

    if filter_cfg is None:
        print(f"[{exp_name}] No filter. Selected all runs: {n_runs}/{n_runs} (100.00%).")
        return list(range(n_runs))

    fkey = _get_filter_key(filter_cfg)
    fmin = filter_cfg.get("min", None)
    fmax = filter_cfg.get("max", None)
    if fmin is None or fmax is None:
        raise KeyError("filter must contain both 'min' and 'max'.")

    if fkey not in results:
        raise KeyError(f"[{exp_name}] filter_key '{fkey}' not found in results.json.")

    fvals = results[fkey]
    if len(fvals) != n_runs:
        raise ValueError(f"[{exp_name}] filter metric '{fkey}' length mismatch.")

    selected = []
    excluded = []  # list of (idx, value)
    for i, v in enumerate(fvals):
        # handle None / NaN robustly
        if v is None:
            excluded.append((i, v))
            continue
        try:
            fv = float(v)
        except Exception:
            excluded.append((i, v))
            continue
        if np.isnan(fv):
            excluded.append((i, fv))
            continue

        if fmin <= fv <= fmax:
            selected.append(i)
        else:
            excluded.append((i, fv))

    sel_ratio = (len(selected) / n_runs * 100.0) if n_runs > 0 else 0.0
    print(f"[{exp_name}] Filter: key='{fkey}', range=[{fmin}, {fmax}]")
    print(f"[{exp_name}] Selected runs: {len(selected)}/{n_runs} ({sel_ratio:.2f}%)")

    if excluded:
        # Print indices not selected and their filter values
        excluded_str = ", ".join([f"{idx}:{val}" for idx, val in excluded])
        print(f"[{exp_name}] Excluded run indices and {fkey} values -> {excluded_str}")
    else:
        print(f"[{exp_name}] No excluded runs.")

    return selected


def _extract_metric_values(
    results: Dict[str, List[float]],
    metric_key: str,
    selected_idx: List[int],
    exp_name: str,
) -> List[float]:
    if metric_key not in results:
        raise KeyError(f"[{exp_name}] metric_key '{metric_key}' not found in results.json.")

    vals = results[metric_key]
    out = []
    for i in selected_idx:
        v = vals[i]
        if v is None:
            continue
        try:
            fv = float(v)
        except Exception:
            continue
        if np.isnan(fv):
            continue
        out.append(fv)
    return out


def plot_barchart_for_metric(
    exps: List[Dict[str, Any]],
    metric_cfg: Dict[str, Any],
    filter_cfg: Optional[Dict[str, Any]],
    save_path: str,
) -> str:
    """
    Bar chart: mean ± std for each experiment.
    Return saved file path.
    """
    os.makedirs(save_path, exist_ok=True)

    metric_name = metric_cfg.get("name", None)           # y-axis label + filename
    metric_key = metric_cfg["key"]             # key in results.json
    title = metric_cfg.get("title", None)      # figure title (optional)
    yrange = metric_cfg.get("range", None)     # tuple (ymin, ymax) or None

    exp_names = []
    means = []
    stds = []
    ns = []
    colors = []

    for exp in exps:
        exp_name = exp["name"]
        results_path = exp["results_path"]
        colors.append(exp.get("color", None)) 

        results = _load_results_json(results_path)
        _validate_lengths(results, exp_name, results_path)
        selected_idx = _select_run_indices(results, exp_name, filter_cfg)
        values = _extract_metric_values(results, metric_key, selected_idx, exp_name)

        if len(values) == 0:
            m = np.nan
            s = np.nan
        else:
            m = float(np.mean(values))
            # std across runs: use sample std when n>1, else 0
            s = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

        exp_names.append(exp_name)
        means.append(m)
        stds.append(s)
        ns.append(len(values))

    x = np.arange(len(exp_names))
    fig, ax = plt.subplots(figsize=(max(5, len(exp_names) * 1.0), 4.0))

    ax.bar(x, means, yerr=stds, capsize=12, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(exp_names, rotation=20, ha="right")
    if metric_name:
        ax.set_ylabel(metric_name)
    # 让网格线在元素下方（不遮挡柱子）
    ax.set_axisbelow(True)
    # 在每个 y-tick 位置画浅灰色横线
    ax.yaxis.grid(True, which="major", color="lightgray", linestyle="-", linewidth=1, alpha=0.6)

    if title:
        ax.set_title(title)
    if yrange is not None:
        ax.set_ylim(yrange[0], yrange[1]+0.08)

    # Optional: show n under each bar (often helpful). If you don't want it, delete this block.
    # for xi, n in zip(x, ns):
    #     ax.text(xi, 0, f"n={n}", ha="center", va="bottom", fontsize=9, rotation=0)

    fig.tight_layout()
    figure_name = title if title else metric_name if metric_name else metric_key
    fname = f"barchart_{_sanitize_filename(figure_name)}.png"
    out_path = os.path.join(save_path, fname)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return out_path


def plot_boxplot_for_metric(
    exps: List[Dict[str, Any]],
    metric_cfg: Dict[str, Any],
    filter_cfg: Optional[Dict[str, Any]],
    save_path: str,
) -> str:
    """
    Box plot for each experiment (using filtered runs).
    Return saved file path.
    """
    os.makedirs(save_path, exist_ok=True)

    metric_name = metric_cfg["name"]
    metric_key = metric_cfg["key"]
    title = metric_cfg.get("title", None)
    yrange = metric_cfg.get("range", None)

    exp_names = []
    data = []

    for exp in exps:
        exp_name = exp["name"]
        results_path = exp["results_path"]

        results = _load_results_json(results_path)
        _validate_lengths(results, exp_name, results_path)
        selected_idx = _select_run_indices(results, exp_name, filter_cfg)
        values = _extract_metric_values(results, metric_key, selected_idx, exp_name)

        exp_names.append(exp_name)
        data.append(values)

    fig, ax = plt.subplots(figsize=(max(5, len(exp_names) * 1.0), 4.0))

    ax.boxplot(data, labels=exp_names, showfliers=True)
    ax.set_ylabel(metric_name)
    # 在每个 y-tick 位置画浅灰色横线
    ax.yaxis.grid(True, which="major", color="lightgray", linestyle="-", linewidth=1, alpha=0.6)

    if title:
        ax.set_title(title)
    if yrange is not None:
        ax.set_ylim(yrange[0]-0.08, yrange[1]+0.08)

    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")

    fig.tight_layout()

    fname = f"boxplot_{_sanitize_filename(metric_name)}.png"
    out_path = os.path.join(save_path, fname)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return out_path


def plot_all_metrics(
    exps: List[Dict[str, Any]],
    metric_key_list: List[Dict[str, Any]],
    filter_cfg: Optional[Dict[str, Any]],
    save_path: str,
) -> Dict[str, Dict[str, str]]:
    """
    For each metric in metric_key_list, generate:
      - bar chart (mean±std)
      - box plot
    Return a dict mapping metric_name -> {'barchart': path, 'boxplot': path}
    """
    os.makedirs(save_path, exist_ok=True)

    outputs = {}
    for metric_cfg in metric_key_list:
        metric_name = metric_cfg["name"]
        print("\n" + "=" * 80)
        print(f"Plotting metric: {metric_name} (key='{metric_cfg['key']}')")

        bar_path = plot_barchart_for_metric(exps, metric_cfg, filter_cfg, save_path)
        box_path = plot_boxplot_for_metric(exps, metric_cfg, filter_cfg, save_path)

        outputs[metric_name] = {"barchart": bar_path, "boxplot": box_path}
        print(f"Saved: {bar_path}")
        print(f"Saved: {box_path}")

    return outputs


## Chinese-tally

In [2]:
exps = [
    {"name": "ELIPS", "color": "#88cc88", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.19_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_ZHENG_tripleSet_Fullsymm/PIPELINE_EVAL/all_results.json"},
    {"name": "w/o symm.", "color": "#ff8888", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.19_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_ZHENG_tripleSet_Nothing/PIPELINE_EVAL/all_results.json"},
    {"name": "Recon. only", "color": "#ffcc66", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.19_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_ZHENG_tripleSet_PureVQ/PIPELINE_EVAL/all_results.json"},
    {"name": "Ceiling", "color": "#88aaff", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.19_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_ZHENG_tripleSet_Nothing_trainAll/PIPELINE_EVAL/all_results.json"},
]
metric_key_list = [
    {"name": "Addition accuracy (HM)", "key": "eval_set_one2n_accu", "title": None, "range": (0, 1)},
    {"name": "Addition accuracy (MD)", "key": "eval_set_emb_select_accu", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (HM)", "key": "train_interpolate_modedict", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (MD)", "key": "train_interpolate_selection", "title": None, "range": (0, 1)},
    {"name": "Orderliness", "key": "orderliness", "title": None, "range": (0.7, 1)},
    {"name": "Substraction", "key": "substraction_one", "title": None, "range": (0.1, 0.7)},
]
# filter can be None
# filter_cfg = {"filter_key": "metric_key1", "min": -1, "max": 2}
filter_cfg = None
save_path = "/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng"
plot_all_metrics(exps, metric_key_list, filter_cfg, save_path)


Plotting metric: Addition accuracy (HM) (key='eval_set_one2n_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Addition_accuracy__HM_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Addition_accuracy__HM_.png

Plotting metric: Addition accuracy (MD) (key='eval_set_emb_select_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20

/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Addition_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Addition_accuracy__MD_.png

Plotting metric: Intepolation accuracy (HM) (key='train_interpolate_modedict')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Intepolation_accuracy__HM_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Intepolation_accuracy__HM_.png

Plotting metric: Intepolation accuracy (MD) (key='train_interpolate_selection')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


...

Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Intepolation_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Intepolation_accuracy__MD_.png

Plotting metric: Orderliness (key='orderliness')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Orderliness.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Orderliness.png

Plotting metric: Substraction (key='substraction_one')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Substraction.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Substraction.png


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


{'Addition accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Addition_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Addition_accuracy__HM_.png'},
 'Addition accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Addition_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Addition_accuracy__MD_.png'},
 'Intepolation accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Intepolation_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Intepolation_accuracy__HM_.png'},
 'Intepolation accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Intepolation_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/boxplot_Intepolation_accuracy__MD_.png'},
 'Orderliness': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/zheng/barchart_Orderliness.png',

## EU-tally

In [3]:
exps = [
    {"name": "ELIPS", "color": "#88cc88", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.22_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_EUTally_tripleSet_Fullsymm/PIPELINE_EVAL/all_results.json"},
    {"name": "w/o symm.", "color": "#ff8888", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.22_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_EUTally_tripleSet_Nothing/PIPELINE_EVAL/all_results.json"},
    {"name": "Recon. only", "color": "#ffcc66", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.22_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_EUTally_tripleSet_PureVQ/PIPELINE_EVAL/all_results.json"},
    {"name": "Ceiling", "color": "#88aaff", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2024.05.22_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_EUTally_tripleSet_Nothing_trainAll/PIPELINE_EVAL/all_results.json"},
]
metric_key_list = [
    {"name": "Addition accuracy (HM)", "key": "eval_set_one2n_accu", "title": None, "range": (0, 1)},
    {"name": "Addition accuracy (MD)", "key": "eval_set_emb_select_accu", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (HM)", "key": "train_interpolate_modedict", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (MD)", "key": "train_interpolate_selection", "title": None, "range": (0, 1)},
    {"name": "Orderliness", "key": "orderliness", "title": None, "range": (0.7, 1)},
    {"name": "Substraction", "key": "substraction_one", "title": None, "range": (0.1, 0.7)},
]
# filter can be None
# filter_cfg = {"filter_key": "metric_key1", "min": -1, "max": 2}
filter_cfg = None
save_path = "/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally"
plot_all_metrics(exps, metric_key_list, filter_cfg, save_path)


Plotting metric: Addition accuracy (HM) (key='eval_set_one2n_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Addition_accuracy__HM_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Addition_accuracy__HM_.png

Plotting metric: Addition accuracy (MD) (key='eval_set_emb_select_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Addition_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Addition_accuracy__MD_.png

Plotting metric: Intepolation accuracy (HM) (key='train_interpolate_modedict')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


...

Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Intepolation_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Intepolation_accuracy__MD_.png

Plotting metric: Orderliness (key='orderliness')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Orderliness.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Orderliness.png

Plotting metric: Substraction (key='substraction_one')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Substraction.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Substraction.png


/tmp/ipykernel_2783490/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


{'Addition accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Addition_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Addition_accuracy__HM_.png'},
 'Addition accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Addition_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Addition_accuracy__MD_.png'},
 'Intepolation accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Intepolation_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Intepolation_accuracy__HM_.png'},
 'Intepolation accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/barchart_Intepolation_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally/boxplot_Intepolation_accuracy__MD_.png'},
 'Orderliness': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/EU-tally

## sing-style 蓝点加糊

In [4]:
exps = [
    {"name": "ELIPS", "color": "#88cc88", "results_path": "/home/zc/mml/z_c_1/S3Plus/VQ/exp/2025.06.13_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_tripleSet_Fullsymm_OnlineBlur/PIPELINE_EVAL/all_results.json"},
    {"name": "w/o symm.", "color": "#ff8888", "results_path": "/home/zc/mml/z_c_1/S3Plus/VQ/exp/2025.06.16_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_tripleSet_Nothing_OnlineBlur/PIPELINE_EVAL/all_results.json"},
    {"name": "Recon. only", "color": "#ffcc66", "results_path": "/home/zc/mml/z_c_1/S3Plus/VQ/exp/2025.06.16_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_tripleSet_PureVQ_OnlineBlur/PIPELINE_EVAL/all_results.json"},
    {"name": "Ceiling", "color": "#88aaff", "results_path": "/home/zc/mml/z_c_1/S3Plus/VQ/exp/2025.06.13_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_tripleSet_Nothing_trainAll_OnlineBlur/PIPELINE_EVAL/all_results.json"},
]
metric_key_list = [
    {"name": "Addition accuracy (HM)", "key": "eval_set_one2n_accu", "title": None, "range": (0, 1)},
    {"name": "Addition accuracy (MD)", "key": "eval_set_emb_select_accu", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (HM)", "key": "train_interpolate_modedict", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (MD)", "key": "train_interpolate_selection", "title": None, "range": (0, 1)},
    {"name": "Orderliness", "key": "orderliness", "title": None, "range": (0.0, 1)},
    {"name": "Substraction", "key": "substraction_one", "title": None, "range": (0.0, 0.7)},
]
# filter can be None
# filter_cfg = {"filter_key": "metric_key1", "min": -1, "max": 2}
filter_cfg = None
save_path = "Blur_dots_blur"
plot_all_metrics(exps, metric_key_list, filter_cfg, save_path)


Plotting metric: Addition accuracy (HM) (key='eval_set_one2n_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_823978/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: Blur_dots_blur/barchart_Addition_accuracy__HM_.png
Saved: Blur_dots_blur/boxplot_Addition_accuracy__HM_.png

Plotting metric: Addition accuracy (MD) (key='eval_set_emb_select_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_823978/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: Blur_dots_blur/barchart_Addition_accuracy__MD_.png
Saved: Blur_dots_blur/boxplot_Addition_accuracy__MD_.png

Plotting metric: Intepolation accuracy (HM) (key='train_interpolate_modedict')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: Blur_dots_blur/barchart_Intepolation_accuracy__HM_.png
Saved: Blur_dots_blur/boxplot_Intepolation_accuracy__HM_.png

Plotting metric: Intepolation accuracy (MD) (key='train_interpolate_selection')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter

...

[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: Blur_dots_blur/barchart_Orderliness.png
Saved: Blur_dots_blur/boxplot_Orderliness.png

Plotting metric: Substraction (key='substraction_one')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_823978/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: Blur_dots_blur/barchart_Substraction.png
Saved: Blur_dots_blur/boxplot_Substraction.png


/tmp/ipykernel_823978/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


{'Addition accuracy (HM)': {'barchart': 'Blur_dots_blur/barchart_Addition_accuracy__HM_.png',
  'boxplot': 'Blur_dots_blur/boxplot_Addition_accuracy__HM_.png'},
 'Addition accuracy (MD)': {'barchart': 'Blur_dots_blur/barchart_Addition_accuracy__MD_.png',
  'boxplot': 'Blur_dots_blur/boxplot_Addition_accuracy__MD_.png'},
 'Intepolation accuracy (HM)': {'barchart': 'Blur_dots_blur/barchart_Intepolation_accuracy__HM_.png',
  'boxplot': 'Blur_dots_blur/boxplot_Intepolation_accuracy__HM_.png'},
 'Intepolation accuracy (MD)': {'barchart': 'Blur_dots_blur/barchart_Intepolation_accuracy__MD_.png',
  'boxplot': 'Blur_dots_blur/boxplot_Intepolation_accuracy__MD_.png'},
 'Orderliness': {'barchart': 'Blur_dots_blur/barchart_Orderliness.png',
  'boxplot': 'Blur_dots_blur/boxplot_Orderliness.png'},
 'Substraction': {'barchart': 'Blur_dots_blur/barchart_Substraction.png',
  'boxplot': 'Blur_dots_blur/boxplot_Substraction.png'}}

## Multi-style 各色点状图案

In [4]:
exps = [
    {"name": "ELIPS", "color": "#88cc88", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2025.05.15_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_multiStyle_Fullsymm/PIPELINE_EVAL/all_results.json"},
    {"name": "w/o symm.", "color": "#ff8888", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2025.05.15_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_multiStyle_Nothing/PIPELINE_EVAL/all_results.json"},
    {"name": "Recon. only", "color": "#ffcc66", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2025.06.10_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_multiStyle_PureVQ/PIPELINE_EVAL/all_results.json"},
    {"name": "Ceiling", "color": "#88aaff", "results_path": "/nfs-stor/xuanjie.liu/S3Plus/VQ/exp/2025.05.19_10vq_Zc[2]_Zs[0]_edim1_[0-20]_plus1024_1_multiStyle_Nothing_trainAll/PIPELINE_EVAL/all_results.json"},
]
metric_key_list = [
    {"name": "Addition accuracy (HM)", "key": "eval_set_one2n_accu", "title": None, "range": (0, 1)},
    {"name": "Addition accuracy (MD)", "key": "eval_set_emb_select_accu", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (HM)", "key": "train_interpolate_modedict", "title": None, "range": (0, 1)},
    {"name": "Intepolation accuracy (MD)", "key": "train_interpolate_selection", "title": None, "range": (0, 1)},
    {"name": "Orderliness", "key": "orderliness", "title": None, "range": (0.0, 1)},
    {"name": "Substraction", "key": "substraction_one", "title": None, "range": (0.1, 0.7)},
]
# filter can be None
# filter_cfg = {"filter_key": "metric_key1", "min": -1, "max": 2}
filter_cfg = None
save_path = "/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots"
plot_all_metrics(exps, metric_key_list, filter_cfg, save_path)


Plotting metric: Addition accuracy (HM) (key='eval_set_one2n_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_1420664/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Addition_accuracy__HM_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Addition_accuracy__HM_.png

Plotting metric: Addition accuracy (MD) (key='eval_set_emb_select_accu')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_1420664/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Addition_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Addition_accuracy__MD_.png

Plotting metric: Intepolation accuracy (HM) (key='train_interpolate_modedict')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


...

Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Intepolation_accuracy__MD_.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Intepolation_accuracy__MD_.png

Plotting metric: Orderliness (key='orderliness')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).


/tmp/ipykernel_1420664/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Orderliness.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Orderliness.png

Plotting metric: Substraction (key='substraction_one')
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
[ELIPS] No filter. Selected all runs: 20/20 (100.00%).
[w/o symm.] No filter. Selected all runs: 20/20 (100.00%).
[Recon. only] No filter. Selected all runs: 20/20 (100.00%).
[Ceiling] No filter. Selected all runs: 20/20 (100.00%).
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Substraction.png
Saved: /nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Substraction.png


/tmp/ipykernel_1420664/2063537075.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=exp_names, showfliers=True)


{'Addition accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Addition_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Addition_accuracy__HM_.png'},
 'Addition accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Addition_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Addition_accuracy__MD_.png'},
 'Intepolation accuracy (HM)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Intepolation_accuracy__HM_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Intepolation_accuracy__HM_.png'},
 'Intepolation accuracy (MD)': {'barchart': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/barchart_Intepolation_accuracy__MD_.png',
  'boxplot': '/nfs-stor/xuanjie.liu/S3Plus/VQ/plots/Multi-style-dots/boxplot_Intepolation_accuracy__MD_.png'},
 'Orderlines